In [46]:
import pandas as pd
import numpy as np

In [47]:
data_df = pd.read_csv("typology_dataset.csv", encoding="utf-8")
cleaned_df = data_df[["id", "account", "full_name", "biography",  "is_verified", "followers", "url", "posts", "category", "political", "typology", "typology_justification", "typology_certainty", "account_summary"]].copy()

In [48]:
print(cleaned_df.shape[0])
cleaned_df.head(1)

2472


,id,account,full_name,biography,is_verified,followers,url,posts,category,political,typology,typology_justification,typology_certainty,account_summary
0,628162163.0,anastasiagerrans,Anastasia Gerrans,new york city\nanastasia@famousprojects.com,True,1079599,https://www.instagram.com/anastasiagerrans/,"[{'caption': ""If you told my 14 year old self ...",fashion & beauty,non-political,not applicable,The account does not discuss any political con...,NaN,"The account focuses on fashion and beauty, sha..."


In [49]:
cleaned_df["typology"].value_counts()

typology
not applicable                1744
loyal liberal                  256
leftward progressive           234
no apologies right              96
tuned-out middle                49
pragmatic and polite right      47
left-out left                   23
UNKNOWN                         13
faith first conservative         8
unconventional right             2
Name: count, dtype: int64

In [50]:
news_df = cleaned_df[cleaned_df["category"] == "news"].copy()
news_df["typology"].value_counts()

typology
loyal liberal                 203
leftward progressive          121
not applicable                 75
no apologies right             52
tuned-out middle               28
pragmatic and polite right     27
left-out left                   8
faith first conservative        1
Name: count, dtype: int64

In [51]:
students = [
    "sb132", "cd111", "ac163", "ct114", "os103",
    "rp104", "gb107", "ep123", "km124", "am149",
    "il115", "ec132", "ms162", "th118", "al141"
]

In [52]:
import pandas as pd

# 1. Load original cleaned dataset
df = pd.read_csv("typology_dataset.csv", encoding="utf-8")

# 2. Define student assignments per batch (6 pairs + 1 triplet = 15 students total)
student_pairs = {
    1: ("sb132", "cd111"),
    2: ("ac163", "ct114"),
    3: ("os103", "rp104"),
    4: ("gb107", "ep123"),
    5: ("km124", "am149"),
    6: ("il115", "ec132"),
    7: ("ms162", "th118", "al141"),  # Triplet batch for the 15th student
}

# 3. Exact category targets per batch (16 posts per batch)
batch_targets = {
    1: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 1, "faith first conservative": 0, "unconventional right": 1, "not applicable": 4},
    2: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 1, "faith first conservative": 0, "unconventional right": 1, "not applicable": 4},
    3: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 1, "faith first conservative": 1, "unconventional right": 0, "not applicable": 4},
    4: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 0, "faith first conservative": 2, "unconventional right": 0, "not applicable": 4},
    5: {"loyal liberal": 6, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 0, "faith first conservative": 1, "unconventional right": 0, "not applicable": 4},
    6: {"loyal liberal": 6, "leftward progressive": 4, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 0, "faith first conservative": 2, "unconventional right": 0, "not applicable": 4},
    7: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 0, "left-out left": 1, "faith first conservative": 2, "unconventional right": 0, "not applicable": 4},
}

# 4. Stratified sampling with "news" prioritization and random shuffling
sampled_rows = []

for category in df["typology"].unique():
    cat_df = df[df["typology"] == category]

    # Check for "news" category (case-insensitive & handles missing values)
    is_news = cat_df["category"].fillna("").astype(str).str.lower() == "news"

    # Randomly shuffle news accounts and non-news accounts independently
    news_df = cat_df[is_news].sample(frac=1, random_state=42) if not cat_df[is_news].empty else pd.DataFrame()
    non_news_df = cat_df[~is_news].sample(frac=1, random_state=42) if not cat_df[~is_news].empty else pd.DataFrame()

    # Prioritize news first, followed by non-news
    prioritized_cat_df = pd.concat([news_df, non_news_df], ignore_index=True)

    idx_ptr = 0
    for batch_num in range(1, 8):
        count_needed = batch_targets[batch_num].get(category, 0)
        if count_needed > 0:
            batch_sample = prioritized_cat_df.iloc[idx_ptr : idx_ptr + count_needed].copy()
            batch_sample["batch"] = f"Batch {batch_num}"
            sampled_rows.append(batch_sample)
            idx_ptr += count_needed

sampled_df = pd.concat(sampled_rows, ignore_index=True)

# 5. Assign to student groups with randomized profile display order per batch
final_records = []
for batch_num in range(1, 8):
    # Randomly shuffle items within each batch so profiles appear in random order
    batch_data = sampled_df[sampled_df["batch"] == f"Batch {batch_num}"].sample(frac=1, random_state=42)
    assigned_students = student_pairs[batch_num]

    for _, row in batch_data.iterrows():
        for student_id in assigned_students:
            row_dict = row.to_dict()
            row_dict["user"] = student_id
            final_records.append(row_dict)

# 6. Save updated dataset
final_df = pd.DataFrame(final_records)
final_df.to_csv("annotation_dataset.csv", index=False)